# Autoencoder Eğitimi (Açılarla)
Bu notebook, sağ kolun kaldırma ve indirme hareketlerini analiz etmek için bir Autoencoder modeli eğitmek ve test etmek amacıyla düzenlenmiştir.

## 2. Kütüphane Tanımlama, Yollar ve Parametreler
Gerekli kütüphaneler ve dosya yolları tanımlanır. Ayrıca, model için sabit parametreler belirlenir.

In [1]:
# ——— İMPORTLAR ———
# Gerekli kütüphaneleri projeye dahil ediyoruz
import os, json, math                  # Dosya ve matematik işlemleri
import numpy as np                     # Sayısal işlemler ve dizi yönetimi
import cv2                             # OpenCV: Görüntü işleme
import mediapipe as mp                 # MediaPipe: İnsan iskeleti/poz tespiti
import torch                           # PyTorch: Derin öğrenme framework'ü
import torch.nn as nn                  # PyTorch sinir ağı modülleri
from torch.utils.data import DataLoader, TensorDataset  # Veri yükleme ve batch yönetimi


# Projenin ana klasör yolunu tanımlıyoruz
BASE = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"

# Eğitim ve test için latent vektörlerin, npy çıktılarının ve test videolarının klasör yolları
DIR_LATENT_TRAIN = os.path.join(BASE, "latent_vector")       # Eğitim latent vektörleri
DIR_LATENT_TEST  = os.path.join(BASE, "latent_vector_test")  # Test latent vektörleri
DIR_NPY_TRAIN    = os.path.join(BASE, "npy_cikti")           # Eğitim npy dosyaları (MediaPipe koordinatları)
DIR_NPY_TEST     = os.path.join(BASE, "npy_cikti_test")      # Test npy dosyaları
DIR_TEST_VID     = os.path.join(BASE, "test_video")          # Test videoları

# Yukarıdaki klasörlerin hepsini oluşturuyoruz (mevcutsa hata vermeden geçer)
for d in [DIR_LATENT_TRAIN, DIR_LATENT_TEST, DIR_NPY_TRAIN, DIR_NPY_TEST]:
    os.makedirs(d, exist_ok=True)


# Modelin eğitiminde kullanılacak sabit değerler (hyperparameter)
WINDOW_SIZE   = 30     # Her veri penceresinin uzunluğu (30 frame)
WINDOW_STRIDE = 5      # Pencereler arasında kaç frame kaydırılacağı
LATENT_DIM    = 3      # Autoencoder'ın üreteceği sıkıştırılmış temsil boyutu (latent vektör boyutu)
EPOCHS        = 50     # Eğitimde veri seti üzerinden geçiş sayısı
BATCH_SIZE    = 128    # Her eğitim adımında kullanılacak örnek sayısı
LR            = 1e-3   # Öğrenme oranı (learning rate)
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"  # GPU varsa CUDA, yoksa CPU kullanılır


# Sonuçların her çalışmada aynı olması için random seed ayarlıyoruz
np.random.seed(42)      # NumPy rastgelelik sabitleme
torch.manual_seed(42)   # PyTorch rastgelelik sabitleme

# Bilgilendirme çıktısı
print("Cihaz:", DEVICE)
print("BASE:", BASE)


Cihaz: cpu
BASE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video


## 3. MediaPipe  İskelet çıkarımı ve açı 


In [2]:
# ——— İMPORTLAR ———
import os, json, math  # Dosya yolu, yapılandırma ve temel matematik işlemleri için kullanılır.
import numpy as np     # Sayısal hesaplamalar ve dizi işlemleri için kullanılır.
import cv2             # Video karelerini okumak ve renk uzayı dönüşümleri için kullanılır.
import mediapipe as mp # İnsan pozu (iskelet) tespiti için MediaPipe kütüphanesini kullanır.

# ——— MediaPipe sabitleri ———
mp_pose = mp.solutions.pose                         # MediaPipe Pose modelini kısayol olarak alır.
PL = mp_pose.PoseLandmark                           # Landmark sabit indekslerini pratik kullanım için kısaltır.

# ——— Landmark indeksleri (sağ taraf odaklı) ———
IDX_RS, IDX_RE, IDX_RW = PL.RIGHT_SHOULDER, PL.RIGHT_ELBOW, PL.RIGHT_WRIST  # Dirsek açısı için sağ omuz–dirsek–bileği seçer.
IDX_RH, IDX_LH, IDX_LS = PL.RIGHT_HIP,     PL.LEFT_HIP,    PL.LEFT_SHOULDER # Omuz açısı için sağ kalça–omuz–dirseği ve merkez/ölçek için kalçaları kullanır.

# ——— Yardımcı fonksiyonlar ———
def _ang(v1, v2):
    """İki 2B vektör arasındaki açıyı (0, π) aralığında radyan cinsinden döndürür."""
    a = v1 / (np.linalg.norm(v1) + 1e-8)           # İlk vektörü sayısal kararlılık için küçük bir terimle normalize eder.
    b = v2 / (np.linalg.norm(v2) + 1e-8)           # İkinci vektörü de aynı şekilde normalize eder.
    return float(np.arccos(np.clip(np.dot(a, b), -1.0, 1.0)))  # Noktasal çarpımdan arccosinüs ile açıyı güvenli şekilde hesaplar.

def _joint(a, b, c):
    """A-B-C noktalarından B tepe noktası için eklem açısını hesaplar."""
    return _ang(np.array(a) - np.array(b), np.array(c) - np.array(b))  # B noktasına göre iki yön vektörü oluşturup _ang ile açıyı bulur.

# ——— Ana fonksiyon: videodan açı çıkarımı ———
def extract_angles(video_path):
    """Verilen videodan her kare için [dirsek, omuz] olmak üzere (T,2) boyutlu ve [0,1] aralığına normalize edilmiş açıları döndürür."""
    cap = cv2.VideoCapture(video_path)             # Video dosyasını kare kare okuyacak yakalayıcıyı açar.
    out, last = [], None                           # Çıktı açılarını bir listede tutar ve gerekirse son geçerli açıyı tekrar kullanmak üzere saklar.

    # MediaPipe Pose modelini orta karmaşıklık ve makul eşiklerle başlatır.
    with mp_pose.Pose(model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while True:
            ok, frame = cap.read()                 # Sıradaki kareyi okur ve başarı durumunu kontrol eder.
            if not ok:
                break                              # Kare kalmadıysa döngüden çıkar.

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # MediaPipe beklentisi nedeniyle kareyi BGR'den RGB'ye çevirir.
            res = pose.process(rgb)                # Kare üzerinde iskelet (pose) tespiti yapar.

            if not res.pose_landmarks:             # Hiç landmark bulunamazsa bağlı seriyi koparmamak için son geçerli açıyı tekrarlar.
                if last is not None:
                    out.append(last)               # Sekmeyi (gap) doldurmak için son değeri ekler.
                continue                           # Bir sonraki kareye geçer.

            lm = res.pose_landmarks.landmark       # Tespit edilen tüm landmark'ların listesini alır.

            cx = (lm[IDX_LH].x + lm[IDX_RH].x) / 2.0  # Normalizasyon için merkez olarak iki kalçanın orta noktasını kullanır.
            cy = (lm[IDX_LH].y + lm[IDX_RH].y) / 2.0  # Merkezin y bileşenini aynı şekilde hesaplar.
            scale = math.hypot(lm[IDX_LS].x - lm[IDX_RS].x, lm[IDX_LS].y - lm[IDX_RS].y)  # Pozu ölçeklemek için iki omuz arası mesafeyi kullanır.

            if scale < 1e-6:                       # Ölçek çok küçükse (hatalı tespit) son geçerli değeri ekleyip kareyi atlar.
                if last is not None:
                    out.append(last)               # Gürültülü karelerde sürekliliği korur.
                continue                           # Bir sonraki kareye geçer.

            def P(idx):
                """Belirli bir landmark'ı merkez-ölçek normalizasyonu ile 2B koordinata çevirir."""
                return np.array([(lm[idx].x - cx) / scale, (lm[idx].y - cy) / scale], dtype=np.float32)  # Kişi konumu ve boyutundan bağımsızlaştırır.

            RS, RE, RW, RH = P(IDX_RS), P(IDX_RE), P(IDX_RW), P(IDX_RH)  # Gerekli eklem noktalarının normalize koordinatlarını çıkarır.

            elbow    = _joint(RS, RE, RW) / np.pi  # Dirsek açısını radyan cinsinden hesaplayıp π'ye bölerek [0,1] aralığına normalizer.
            shoulder = _joint(RH, RS, RE) / np.pi  # Omuz açısını da aynı şekilde hesaplayıp normalize eder.

            last = np.array([elbow, shoulder], dtype=np.float32)  # Son geçerli çifti günceller.
            out.append(last)                       # Bu kareye ait açı çiftini çıktıya ekler.

    cap.release()                                  # Video kaynağını serbest bırakır.
    return np.stack(out) if out else np.empty((0, 2), dtype=np.float32)  # Varsa (T,2) numpy dizisi döndürür, yoksa boş (0,2) dizi verir.


## 4. Açısal Veriyi Tüm Frame'ler İçin Uygulama
BASE klasöründeki 60 adet videoyu sırayla işleyerek her karedeki normalize edilmiş dirsek ve omuz açılarını çıkarmak, bu açıları NumPy .npy dosyaları olarak npy_cikti klasörüne kaydetmek ve işlem sürecinde eksik veya başarısız videoları raporlamaktır.

In [3]:
# 7. Açısal Veriyi Tüm Frame’ler için Uygulama
# Girdi:  BASE\1.mp4 ... BASE\60.mp4  → İşlenecek ham videolar
# Çıktı:  BASE\npy_cikti\angles_{id}.npy  → Her video için (T, 2) açısal veri dosyası

saved, total_frames = 0, 0        # Kaydedilen video sayısı ve toplam frame sayısı sayaçlarını başlatır.
missing = []                      # Eksik video yollarını saklamak için liste oluşturur.

for vid in range(1, 61):          # 1’den 60’a kadar her video numarası için döngü başlatır.
    vpath = os.path.join(BASE, f"{vid}.mp4")  # Mevcut video dosyasının tam yolunu oluşturur.
    if not os.path.exists(vpath): # Video dosyası yoksa...
        missing.append(vpath)     # Eksikler listesine ekler.
        continue                  # Döngünün bu turunu atlar.

    ang = extract_angles(vpath)   # Her frame için [dirsek, omuz] açılarını çıkarır (T, 2) matris.
    if ang.shape[0] == 0:         # Eğer hiç frame işlenememişse...
        print(f"[UYARI] Pose bulunamadı veya çok az frame: {vid}.mp4")  # Uyarı mesajı verir.
        continue                  # Döngünün bu turunu atlar.

    out_path = os.path.join(DIR_NPY_TRAIN, f"angles_{vid}.npy")  # Çıktı dosya yolunu oluşturur.
    np.save(out_path, ang)        # Açısal veriyi .npy formatında kaydeder.

    saved += 1                    # Başarıyla işlenen video sayısını artırır.
    total_frames += ang.shape[0]  # Toplam frame sayısına ekler.

print(f"İşlenen video sayısı: {saved}/60  | Toplam frame: {total_frames}")  # Genel özet bilgisi verir.
if missing:                       # Eksik video varsa...
    print("Eksik video dosyaları:")  # Başlık yazdırır.
    for p in missing: print(" -", p) # Eksik video yollarını listeler.


İşlenen video sayısı: 60/60  | Toplam frame: 11976


## 5. Window Size ile Pencereleme

önceden kaydedilmiş açı verilerini (angles_*.npy) alıp

Pencereleme yöntemi ile her veri serisini sabit uzunlukta zaman dilimlerine bölmek,

Bu pencereleri düzleştirerek (flatten) modele uygun giriş formatına getirmek,

Tüm videolardan gelen pencereleri tek bir eğitim matrisi (X_train) halinde birleştirmek.

Böylece her satır bir hareket segmentini temsil eden, model (ör. autoencoder) ile doğrudan kullanılabilecek sabit boyutlu bir veri seti elde edilmiş olur.

In [4]:
# --- Pencereleme yardımcıları ---
def windowize(data: np.ndarray, w: int = WINDOW_SIZE, stride: int = WINDOW_STRIDE) -> np.ndarray:
    """
    (T, D) → (N, w, D) boyutuna pencereleme yapar; kısa dizilerde son frame tekrarlanarak pad edilir.
    """
    if data.ndim != 2 or data.shape[0] == 0:  # Veri 2 boyutlu değilse veya boşsa boş matris döndür.
        return np.empty((0, w, data.shape[1] if data.ndim==2 else 2), dtype=np.float32)

    T, D = data.shape                         # T = zaman (frame sayısı), D = özellik boyutu (2: dirsek, omuz)
    if T < w:                                 # Eğer toplam frame sayısı pencere boyutundan küçükse...
        pad = np.repeat(data[-1:, :], w - T, axis=0)  # Son frame’i yeterli sayıda tekrarlayarak doldur.
        data = np.vstack([data, pad])         # Orijinal veri ile pad edilmiş veriyi birleştir.
        T = data.shape[0]                     # Yeni uzunluğu güncelle.

    wins = []                                 # Pencerelerin saklanacağı liste.
    for start in range(0, T - w + 1, stride): # Belirlenen stride ile pencere başlangıçlarını dolaş.
        wins.append(data[start:start + w])    # Her pencereyi listeye ekle.
    return np.array(wins, dtype=np.float32) if wins else np.empty((0, w, D), dtype=np.float32)  # Pencereleri numpy array olarak döndür.

def flatten_windows(win: np.ndarray) -> np.ndarray:
    """(N, w, D) → (N, w*D) formatına dönüştürür (pencereleri düzleştirir)."""
    return win.reshape(win.shape[0], -1).astype(np.float32) if win.size else np.empty((0, WINDOW_SIZE*2), np.float32)

# --- Eğitim seti için X_train oluştur ---
bags = []                                     # Tüm pencereleri toplayacağımız liste.
missing = []                                  # Eksik dosya yollarını saklayacağımız liste.
total_wins = 0                                # Toplam pencere sayısını tutacak sayaç.

for vid in range(1, 61):                      # 1’den 60’a kadar tüm videolar için döngü.
    fpath = os.path.join(DIR_NPY_TRAIN, f"angles_{vid}.npy")  # Açı verisinin dosya yolunu oluştur.
    if not os.path.exists(fpath):             # Dosya yoksa eksikler listesine ekle ve devam et.
        missing.append(fpath); continue

    ang = np.load(fpath)                      # (T, 2) boyutunda açı verisini yükle.
    wins = windowize(ang)                     # (N, WINDOW_SIZE, 2) formatında pencerelere ayır.
    flat = flatten_windows(wins)              # (N, WINDOW_SIZE*2) formatına düzleştir.
    if flat.shape[0]:                          # Pencere varsa...
        bags.append(flat)                     # Eğitim listesine ekle.
        total_wins += flat.shape[0]            # Toplam pencere sayısını güncelle.
    else:
        print(f"[UYARI] Pencere yok: angles_{vid}.npy (T={ang.shape[0]})")  # Uyarı mesajı yaz.

X_train = np.concatenate(bags, axis=0) if bags else np.empty((0, WINDOW_SIZE*2), np.float32)  # Tüm pencereleri tek bir matris halinde birleştir.
print(f"X_train şekli: {X_train.shape}  | Toplam pencere: {total_wins}")  # Eğitim veri setinin boyutunu yazdır.
if missing:                                     # Eksik dosya varsa listeler.
    print("Eksik açı dosyaları:")
    for p in missing: print(" -", p)


X_train şekli: (2070, 60)  | Toplam pencere: 2070


## 6. Autoencoder Modelinin Tanımı ve Eğitimi
Autoencoder modelini tanımlıyoruz ve eğitim sürecini başlatıyoruz.

### 6.1 Dataset Sınıfı
Doğrulama yapmak (pencere boyutu doğru mu, veri boş mu diye kontrol),

NumPy → PyTorch Tensör dönüşümü yapmak,

Dataset ve DataLoader oluşturmak (mini-batch eğitim için),

GPU  optimizasyonu ile veriyi hızlı yüklemek.

 Bu adım sayesinde model eğitimine başlamadan önce veri, PyTorch’un batch’li şekilde işleyebileceği formata getirilmiş oluyor.

In [5]:
# X_train: (N_pencere, WINDOW_SIZE*2) → bir önceki hücrede üretildi.
assert isinstance(WINDOW_SIZE, int) and WINDOW_SIZE > 0  # Pencere boyutunun pozitif tam sayı olduğunu doğrular.
INPUT_DIM = WINDOW_SIZE * 2  # Her pencere için toplam giriş boyutunu hesaplar (örn: 30*2 = 60).

if X_train.shape[0] == 0:  # Eğitim veri seti boşsa...
    raise RuntimeError("X_train boş. 7.1 pencereleme adımını kontrol et.")  # Kullanıcıyı eksik veri konusunda uyarır.

# Tensör veri kümesi + yükleyici
tensor_X = torch.from_numpy(X_train.astype(np.float32))  # NumPy verisini PyTorch tensörüne çevirir.
ds = TensorDataset(tensor_X)  # Tensörleri PyTorch Dataset formatına sarar.
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
                pin_memory=(DEVICE=='cuda'))  # Dataset'i eğitimde kullanılmak üzere DataLoader ile hazırlar.

print(f"Dataset örnek sayısı: {len(ds)} | Girdi boyutu: {INPUT_DIM}")  # Eğitim veri kümesinin boyutunu ve giriş boyutunu yazdırır.


Dataset örnek sayısı: 2070 | Girdi boyutu: 60


### 6.2 Autoencoder Modeli
Pencereleme ile oluşturulmuş açısal veri setini kullanarak bir Autoencoder  modeli tanımlamak.

Encoder kısmı, giriş verisini daha küçük bir latent vektöre sıkıştırır (boyut indirgeme).

Decoder kısmı, bu latent vektörden giriş verisini tekrar üretmeye çalışır (yeniden inşa).

MSE kaybı ile giriş ve çıkış arasındaki farkı minimize eder, böylece model verinin kompakt ve anlamlı bir temsilini öğrenir.

Bu latent vektörler, benzerlik ölçümü (DTW, cosine similarity vb.) veya hareket analizi için kullanılacak temel özelliklerdir.



In [6]:
class AngleAE(nn.Module):
    """
    Basit ve stabil Autoencoder:
      Encoder: 60 -> 64 -> 16 -> LATENT
      Decoder: LATENT -> 16 -> 64 -> 60
    ReLU aktivasyonları ile kompakt (sıkıştırılmış) temsil öğrenir.
    """
    def __init__(self, d_in=INPUT_DIM, d_lat=LATENT_DIM):
        super().__init__()  # PyTorch nn.Module yapısını başlatır.

        # --- Encoder ---
        self.enc = nn.Sequential(
            nn.Linear(d_in, 64), nn.ReLU(),  # Girişi 64 boyuta çıkarır, ReLU ile doğrusal olmayan dönüşüm yapar.
            nn.Linear(64, 16),   nn.ReLU(),  # 64 → 16 boyuta indirger, ReLU aktivasyonu uygular.
            nn.Linear(16, d_lat)              # 16 → LATENT boyutuna indirger (kompakt temsil).
        )

        # --- Decoder ---
        self.dec = nn.Sequential(
            nn.Linear(d_lat, 16), nn.ReLU(),  # Latent boyutunu tekrar 16’ya çıkarır.
            nn.Linear(16, 64),    nn.ReLU(),  # 16 → 64 boyuta genişletir.
            nn.Linear(64, d_in)                # 64 → giriş boyutuna (60) döndürür.
        )

        # Xavier başlatma: Eğitim sırasında ağırlıkları dengeli başlatmak için kullanılır.
        for m in self.modules():
            if isinstance(m, nn.Linear):               # Sadece Linear katmanlar için uygular.
                nn.init.xavier_uniform_(m.weight)      # Ağırlıkları Xavier uniform ile başlatır.
                nn.init.zeros_(m.bias)                 # Bias değerlerini sıfırlar.

    def forward(self, x):
        z = self.enc(x)  # Girişi encoder’dan geçirerek latent (sıkıştırılmış) vektörü üretir.
        xr = self.dec(z) # Latent vektörden decoder ile girişin yeniden inşasını yapar.
        return xr, z     # Hem yeniden inşa edilen çıkışı hem de latent vektörü döndürür.

# Modeli tanımla ve GPU/CPU’ya gönder.
model = AngleAE(d_in=INPUT_DIM, d_lat=LATENT_DIM).to(DEVICE)

# Adam optimizasyon algoritmasını ayarla.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Kayıp fonksiyonunu MSE (Ortalama Kare Hatası) olarak seç.
criterion = nn.MSELoss()

# Model mimarisini yazdır.
print(model)


AngleAE(
  (enc): Sequential(
    (0): Linear(in_features=60, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=3, bias=True)
  )
  (dec): Sequential(
    (0): Linear(in_features=3, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=60, bias=True)
  )
)


### 6.3 Eğitim Aşamaları

Eğitim öncesi güvenlik kontrolü yapmak (veri boyutu ile model giriş boyutunun uyumlu olduğundan emin olmak),

Eğitim sürecinde en iyi modeli belirlemek için en düşük loss değerini ve o loss’a ait model ağırlıklarını saklamaya hazırlık yapmak.

Böylece model eğitimi sırasında en düşük hatayı veren ağırlıkları kaydedip, eğitim sonunda en iyi modeli kullanabiliriz.

In [7]:
# Basit doğrulamalar
ex = next(iter(dl))[0]  # DataLoader'dan ilk batch'in ilk tensörünü alır.
assert ex.shape[1] == INPUT_DIM, "Girdi boyutu beklenenle eşleşmiyor."  
# Giriş boyutunun (özellik sayısı) modelin beklediği INPUT_DIM ile aynı olduğunu kontrol eder.

# Eğitim istatistikleri
best_loss = float("inf")  # En düşük kaybı (loss) takip etmek için başlangıç değerini sonsuz olarak ayarlar.
best_state = None         # En iyi model ağırlıklarını saklamak için değişken başlatır.


### 6.4 Eğitim Döngüsü

Autoencoder modelini X_train verisi üzerinde belirlenen epoch sayısı kadar eğitmek,

Eğitim sırasında en düşük ortalama kaybı veren ağırlıkları saklamak,

En iyi modeli .pt dosyası olarak kaydetmek,

Modelin ileride doğru şekilde yüklenebilmesi için meta bilgilerini .json formatında saklamak.

Bu adım tamamlandığında elimizde eğitimli autoencoder modeli ve modelin nasıl eğitildiğine dair bilgiler oluyor; bu sayede testi, latent çıkarımı ve benzerlik ölçümü yapılabilir hale geliyoruz.

In [8]:
for epoch in range(1, EPOCHS + 1):  # Belirlenen epoch sayısı kadar eğitim döngüsü başlatır.
    model.train()                   # Modeli eğitim moduna alır (dropout, batchnorm vb. eğitim davranışı için).
    total = 0.0                      # Bu epoch’taki toplam kaybı (loss) saklamak için sayaç başlatır.

    for (batch,) in dl:              # DataLoader üzerinden mini-batch’ler halinde veri alır.
        batch = batch.to(DEVICE, non_blocking=True)  # Batch’i GPU/CPU’ya gönderir.

        optimizer.zero_grad()        # Önceki adımın gradyanlarını sıfırlar.
        recon, _ = model(batch)      # Girdi verisini modelden geçirir, yeniden inşa (reconstruction) alır.
        loss = criterion(recon, batch)  # Giriş ve çıkış arasındaki MSE kaybını hesaplar.
        loss.backward()              # Hata gradyanını hesaplar (geri yayılım).
        optimizer.step()              # Ağırlıkları gradyanlara göre günceller.

        total += loss.item() * batch.size(0)  # Batch kaybını örnek sayısı ile çarparak toplam kayba ekler.

    avg_loss = total / len(ds)       # Epoch ortalama kaybını hesaplar.
    print(f"Epoch {epoch:03d} | Train Loss: {avg_loss:.6f}")  # Epoch bilgilerini ekrana yazdırır.

    # En iyi ağırlıkları sakla (yalnızca train kaybına göre)
    if avg_loss < best_loss:         # Ortalama kayıp önceki en iyiden küçükse...
        best_loss = avg_loss         # En iyi kaybı günceller.
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}  
        # Model ağırlıklarının kopyasını CPU’ya alarak saklar.

# En iyi modeli yükle
if best_state is not None:
    model.load_state_dict(best_state)  # En düşük kayba sahip ağırlıkları modele yükler.

# Modeli dosyaya kaydet
model_path = os.path.join(BASE, "autoencoder_angle.pt")
torch.save(model.state_dict(), model_path)  # Eğitimli modelin ağırlıklarını kaydeder.

# Meta bilgileri kaydet (ileride yüklemek için)
meta = {
    "window_size": WINDOW_SIZE,              # Pencere uzunluğu
    "window_stride": WINDOW_STRIDE,          # Pencere kayma adımı
    "input_dim": INPUT_DIM,                  # Model giriş boyutu
    "latent_dim": LATENT_DIM,                # Latent vektör boyutu
    "features": ["elbow", "shoulder"],       # Kullanılan özellikler (açı sırası)
}
with open(os.path.join(BASE, "autoencoder_angle_meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)  # Meta bilgileri JSON formatında kaydeder.

print(f"Kaydedildi -> {model_path}")  # Modelin kaydedildiğini bildirir.
print(f"En iyi eğitim kaybı: {best_loss:.6f}")  # En düşük eğitim kaybını ekrana yazar.


Epoch 001 | Train Loss: 0.434069
Epoch 002 | Train Loss: 0.206769
Epoch 003 | Train Loss: 0.028930
Epoch 004 | Train Loss: 0.003001
Epoch 005 | Train Loss: 0.000704
Epoch 006 | Train Loss: 0.000194
Epoch 007 | Train Loss: 0.000114
Epoch 008 | Train Loss: 0.000099
Epoch 009 | Train Loss: 0.000096
Epoch 010 | Train Loss: 0.000096
Epoch 011 | Train Loss: 0.000096
Epoch 012 | Train Loss: 0.000095
Epoch 013 | Train Loss: 0.000095
Epoch 014 | Train Loss: 0.000095
Epoch 015 | Train Loss: 0.000094
Epoch 016 | Train Loss: 0.000094
Epoch 017 | Train Loss: 0.000094
Epoch 018 | Train Loss: 0.000094
Epoch 019 | Train Loss: 0.000094
Epoch 020 | Train Loss: 0.000093
Epoch 021 | Train Loss: 0.000092
Epoch 022 | Train Loss: 0.000093
Epoch 023 | Train Loss: 0.000093
Epoch 024 | Train Loss: 0.000092
Epoch 025 | Train Loss: 0.000092
Epoch 026 | Train Loss: 0.000091
Epoch 027 | Train Loss: 0.000091
Epoch 028 | Train Loss: 0.000090
Epoch 029 | Train Loss: 0.000090
Epoch 030 | Train Loss: 0.000091
Epoch 031 

## 7. Her Videoya Ait Latent Vektörlerin Ayrı Kaydedilmesi

Eğitimli autoencoder modelini ve meta bilgilerini yüklemek,

Daha önce çıkarılmış açısal veri (angles_*.npy) dosyalarını pencereleyip modele vermek,

Modelin encoder kısmından latent (sıkıştırılmış) vektörleri elde etmek,

Bu latent vektörleri .npy dosyaları olarak latent_vector klasörüne kaydetmek.

Bu adım tamamlandığında, elimizde her video için hareketin kompakt temsili olan latent vektörler bulunur. Bu vektörler, DTW, cosine similarity veya kümeleme gibi analizlerde kullanılabilir.



In [9]:
import os, json, numpy as np, torch, torch.nn as nn  # Dosya işlemleri, JSON okuma, sayısal işlemler ve PyTorch modülleri.

# ---- Model ve meta bilgilerini yükle ----
meta_path = os.path.join(BASE, "autoencoder_angle_meta.json")  # Modelin meta bilgilerini tutan JSON dosyasının yolu.
weights_path = os.path.join(BASE, "autoencoder_angle.pt")      # Eğitimli model ağırlıklarının bulunduğu dosya yolu.

meta = json.load(open(meta_path, "r"))        # Meta bilgilerini JSON dosyasından yükler.
WINDOW_SIZE   = meta["window_size"]           # Pencere uzunluğunu alır.
WINDOW_STRIDE = meta["window_stride"]         # Pencere kayma adımını alır.
INPUT_DIM     = meta["input_dim"]             # Modelin giriş boyutunu alır.
LATENT_DIM    = meta["latent_dim"]            # Latent (sıkıştırılmış) vektör boyutunu alır.
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"  # GPU varsa CUDA, yoksa CPU seçilir.

# ---- Autoencoder model tanımı ----
class AngleAE(nn.Module):
    def __init__(self, d_in=INPUT_DIM, d_lat=LATENT_DIM):
        super().__init__()
        self.enc = nn.Sequential(              # Encoder: Girişi latent vektöre sıkıştırır.
            nn.Linear(d_in, 64), nn.ReLU(),
            nn.Linear(64, 16),   nn.ReLU(),
            nn.Linear(16, d_lat)
        )
        self.dec = nn.Sequential(              # Decoder: Latent vektörden girişi yeniden oluşturur.
            nn.Linear(d_lat, 16), nn.ReLU(),
            nn.Linear(16, 64),    nn.ReLU(),
            nn.Linear(64, d_in)
        )
    def forward(self, x):
        z = self.enc(x)                        # Encoder’dan latent vektörü çıkarır.
        xr = self.dec(z)                       # Decoder ile yeniden inşa eder.
        return xr, z                           # Hem yeniden inşayı hem de latent vektörü döndürür.

# ---- Modeli yükleme ----
model = AngleAE().to(DEVICE)                   # Modeli oluşturur ve GPU/CPU’ya taşır.
state = torch.load(weights_path, map_location=DEVICE)  # Eğitimli ağırlıkları yükler.
model.load_state_dict(state)                   # Model ağırlıklarını uygular.
model.eval()                                   # Modeli değerlendirme moduna alır.

# ---- Latent üret ve kaydet ----
saved, total_wins = 0, 0                       # Kaydedilen video sayısı ve toplam pencere sayısını tutar.
missing = []                                   # Eksik açı dosyalarının listesi.

for vid in range(1, 61):                       # 1’den 60’a kadar tüm videolar için döngü.
    apath = os.path.join(DIR_NPY_TRAIN, f"angles_{vid}.npy")  # Açısal veri dosya yolunu oluşturur.
    if not os.path.exists(apath):               # Dosya yoksa...
        missing.append(apath)                   # Eksikler listesine ekler.
        continue

    ang = np.load(apath)                        # (T, 2) boyutunda açı verisini yükler.
    wins = windowize(ang, WINDOW_SIZE, WINDOW_STRIDE)  # Veriyi pencerelere böler → (N, W, 2)
    flat = flatten_windows(wins)                # Pencereleri düzleştirir → (N, W*2)

    if flat.shape[0] == 0:                       # Hiç pencere yoksa uyarı verir.
        print(f"[UYARI] Pencere yok: angles_{vid}.npy (T={ang.shape[0]})")
        continue

    with torch.no_grad():                        # Gradyan hesaplamadan inference modunda çalışır.
        tens = torch.from_numpy(flat.astype(np.float32)).to(DEVICE)  # Veriyi tensöre çevirir ve GPU/CPU’ya taşır.
        _, z = model(tens)                       # Modelden latent vektörü çıkarır.
        z_np = z.cpu().numpy()                   # Latent vektörü CPU’ya alıp NumPy array’e çevirir.

    out_path = os.path.join(DIR_LATENT_TRAIN, f"latent_{vid}.npy")  # Latent kaydı için dosya yolu oluşturur.
    np.save(out_path, z_np)                      # Latent vektörü .npy formatında kaydeder.

    saved += 1                                   # Kaydedilen video sayısını artırır.
    total_wins += z_np.shape[0]                  # Toplam pencere sayısını artırır.

# ---- Özet ----
print(f"Latent kaydı tamam: {saved}/60 video | Toplam pencere: {total_wins}")  # İşlem özeti verir.
if missing:                                     # Eksik dosya varsa listeler.
    print("Eksik açı dosyaları:")
    for p in missing: print(" -", p)


Latent kaydı tamam: 60/60 video | Toplam pencere: 2070



## 8. Test Verisinin İşlenmesi ve Kaydı

61–64 numaralı test videolarını işleyip her kare için normalize edilmiş dirsek ve omuz açılarını çıkarmak,

Bu açı verilerini pencerelere bölüp düzleştirerek eğitilmiş autoencoder modeline vermek,

Modelin encoder’ından çıkan latent (sıkıştırılmış) vektörleri .npy dosyalarına kaydetmek,

Böylece test verileri için hem ham açısal veri hem de latent temsil hazır hale getirmek.

 Bu adım, eğitimde yaptığımız işlemlerin test verisine uygulanmış hali olduğu için, artık bu latent vektörleri benzerlik analizi (DTW, cosine similarity vb.) için kullanabiliriz.

In [10]:
import os, json, numpy as np, torch, torch.nn as nn  # Dosya yolları, JSON okuma, sayısal işlemler ve PyTorch modülleri.

# ---- Yol tanımları ----
BASE = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"  # Ana proje klasörü.
DIR_TEST_VID     = os.path.join(BASE, "test_video")            # Test videolarının bulunduğu klasör.
DIR_NPY_TEST     = os.path.join(BASE, "npy_cikti_test")        # Test videoları için açı verilerinin kaydedileceği klasör.
DIR_LATENT_TEST  = os.path.join(BASE, "latent_vector_test")    # Test videolarının latent vektörlerinin kaydedileceği klasör.
os.makedirs(DIR_NPY_TEST, exist_ok=True)                       # Açı çıktısı klasörünü oluştur (varsa hata vermez).
os.makedirs(DIR_LATENT_TEST, exist_ok=True)                    # Latent çıktısı klasörünü oluştur (varsa hata vermez).

# ---- Model + meta bilgilerini yükleme ----
meta = json.load(open(os.path.join(BASE, "autoencoder_angle_meta.json"), "r"))  # Meta bilgilerini JSON'dan oku.
WINDOW_SIZE, WINDOW_STRIDE = meta["window_size"], meta["window_stride"]         # Pencere boyutu ve kayma adımı.
INPUT_DIM, LATENT_DIM = meta["input_dim"], meta["latent_dim"]                   # Model giriş boyutu ve latent boyut.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"                         # GPU varsa CUDA, yoksa CPU kullan.

# ---- Autoencoder model tanımı ----
class AngleAE(nn.Module):
    def __init__(self, d_in=INPUT_DIM, d_lat=LATENT_DIM):
        super().__init__()
        # Encoder: 60 → 64 → 16 → LATENT
        self.enc = nn.Sequential(nn.Linear(d_in,64), nn.ReLU(),
                                 nn.Linear(64,16), nn.ReLU(),
                                 nn.Linear(16,d_lat))
        # Decoder: LATENT → 16 → 64 → 60
        self.dec = nn.Sequential(nn.Linear(d_lat,16), nn.ReLU(),
                                 nn.Linear(16,64), nn.ReLU(),
                                 nn.Linear(64,d_in))
    def forward(self,x):
        z = self.enc(x)       # Latent vektörü çıkar.
        return self.dec(z), z # Yeniden inşa edilmiş veri ve latent vektörü döndür.

# ---- Model yükleme ----
model = AngleAE().to(DEVICE)                                                              # Modeli oluştur ve GPU/CPU’ya gönder.
model.load_state_dict(torch.load(os.path.join(BASE, "autoencoder_angle.pt"), map_location=DEVICE))  # Eğitimli ağırlıkları yükle.
model.eval()                                                                              # Modeli değerlendirme moduna al.

# ---- 61..64 arası test videolarını işleme ----
for vid in range(61, 65):                                                                 # Test için video numaraları.
    vpath = os.path.join(DIR_TEST_VID, f"{vid}.mp4")                                      # Video dosya yolunu oluştur.
    if not os.path.exists(vpath):                                                         # Video yoksa uyarı ver ve atla.
        print(f"[UYARI] Yok: {vpath}")
        continue

    ang = extract_angles(vpath)                                                           # Videodan [dirsek, omuz] açılarını çıkar (T,2).
    np.save(os.path.join(DIR_NPY_TEST, f"angles_{vid}.npy"), ang)                          # Açısal veriyi .npy formatında kaydet.

    wins = windowize(ang, WINDOW_SIZE, WINDOW_STRIDE)                                     # Veriyi pencerelere böl (N,W,2).
    flat = flatten_windows(wins)                                                          # Pencereleri düzleştir (N,W*2).
    if flat.shape[0] == 0:                                                                # Hiç pencere oluşmadıysa uyarı ver.
        print(f"[UYARI] Pencere yok: {vid}")
        continue

    with torch.no_grad():                                                                 # Gradyan hesaplamadan çalış (inference modu).
        tens = torch.from_numpy(flat.astype(np.float32)).to(DEVICE)                       # Veriyi tensöre çevir ve GPU/CPU’ya gönder.
        _, z = model(tens)                                                                # Modelden latent vektörleri al (N, LATENT_DIM).
        np.save(os.path.join(DIR_LATENT_TEST, f"latent_{vid}.npy"), z.cpu().numpy())       # Latent vektörleri .npy formatında kaydet.

print("Test açı/latent dosyaları hazır.")                                                 # İşlem tamam mesajı.


Test açı/latent dosyaları hazır.


## 9. Eğitim Latentlerinin Yüklenmesi
Eğitim veri setindeki her videonun latent vektör dosyalarını (latent_*.npy) yüklemek,

Bu latent vektörleri video kimliği ile eşleştirerek bir sözlükte saklamak,

Eksik dosyaları tespit edip kullanıcıya bildirmek.

Bu adım sayesinde eğitim veri setinin tüm latent temsillerine kolayca erişebiliriz ve bu verilerle karşılaştırma, benzerlik analizi veya sınıflandırma yapabiliriz.

In [11]:
DIR_LATENT_TRAIN = os.path.join(BASE, "latent_vector")  # Eğitim sırasında üretilen latent vektörlerin bulunduğu klasörün yolu.

train_latents = {}  # Eğitim verisi latent vektörlerini saklamak için sözlük (video_id → latent matris).

for vid in range(1, 61):  # 1’den 60’a kadar tüm eğitim videolarını dolaş.
    f = os.path.join(DIR_LATENT_TRAIN, f"latent_{vid}.npy")  # İlgili video için latent dosya yolunu oluştur.
    if os.path.exists(f):  # Dosya mevcutsa...
        train_latents[vid] = np.load(f)  # Latent vektörü NumPy ile yükleyip sözlüğe ekle.
    else:  # Dosya yoksa...
        print(f"[UYARI] Eksik: latent_{vid}.npy")  # Eksik dosya uyarısı ver.

print("Yüklü eğitim latent sayısı:", len(train_latents))  # Kaç adet video için latent vektör yüklendiğini yazdır.


Yüklü eğitim latent sayısı: 60


## 10. Test Verilerinin Modele Uyum Testi
Test videoları, model ile karşılaştırılarak benzerlik yüzdeleri hesaplanır.

Referans seçimi: Eğitim latentlerinden “medoid-benzeri” tek bir dizi seçiyoruz; bu, eğitim dağılımını en iyi temsil eden örnek oluyor.

Ölçekleme: Referans ↔ eğitim DTW uzaklıklarını dağıtım olarak ölçüyoruz; p5 “çok benzer”, p95 “zayıf benzer” sınırı gibi düşünülüyor.

Yüzde hesap: Test DTW’sini bu aralığa yerleştirip 0–100% arası benzerlik üretiyoruz (küçük uzaklık → yüksek puan).

In [12]:
# ====== IMPORT & YOLLAR ======
import os, json, numpy as np, torch, torch.nn as nn  # Dosya, meta, sayısal ve PyTorch işlemleri için gerekli kütüphaneleri yükler.

BASE = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"  # Proje ana klasör yolunu tanımlar.
DIR_TR_ANG = os.path.join(BASE, "npy_cikti")                   # Eğitim açı dosyalarının bulunduğu klasörü tanımlar.
DIR_TE_ANG = os.path.join(BASE, "npy_cikti_test")              # Test açı dosyalarının yazılacağı/okunacağı klasörü tanımlar.
os.makedirs(DIR_TE_ANG, exist_ok=True)                         # Test açı klasörü yoksa oluşturur.

# ====== META & MODEL ======
meta = json.load(open(os.path.join(BASE, "autoencoder_angle_meta.json"), "r"))  # Eğitimde kullanılan meta bilgilerini okur.
WINDOW_SIZE   = meta["window_size"]                                             # Pencere uzunluğunu ayarlar.
WINDOW_STRIDE = meta["window_stride"]                                           # Pencere kayma adımını ayarlar.
INPUT_DIM     = meta["input_dim"]                                               # Modelin giriş boyutunu (W*2) ayarlar.
LATENT_DIM    = meta["latent_dim"]                                              # Latent boyutu bilgisi okunur (burada yalnızca mimari için lazımdır).
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"                  # GPU varsa CUDA yoksa CPU seçer.

class AngleAE(nn.Module):
    """Eğitimdeki AE mimarisi; reconstruction (yeniden üretim) hatasını ölçmek için kullanılır."""
    def __init__(self, d_in=INPUT_DIM, d_lat=LATENT_DIM):
        super().__init__()                                                      # PyTorch modülünü başlatır.
        self.enc = nn.Sequential(nn.Linear(d_in,64), nn.ReLU(),
                                 nn.Linear(64,16),   nn.ReLU(),
                                 nn.Linear(16,d_lat))                           # Girişi latent boyuta sıkıştıran encoder’ı tanımlar.
        self.dec = nn.Sequential(nn.Linear(d_lat,16), nn.ReLU(),
                                 nn.Linear(16,64),    nn.ReLU(),
                                 nn.Linear(64,d_in))                            # Latent’ten girişi yeniden üreten decoder’ı tanımlar.
    def forward(self, x):
        z  = self.enc(x)                                                        # Girişi latent vektöre dönüştürür.
        xr = self.dec(z)                                                        # Latent vektörden girişi yeniden üretir.
        return xr                                                               # Benzerlik hesapları için yalnızca yeniden üretimi döndürür.

model = AngleAE().to(DEVICE)                                                   # Modeli oluşturur ve uygun cihaza taşır.
state = torch.load(os.path.join(BASE, "autoencoder_angle.pt"), map_location=DEVICE)  # Kaydedilmiş en iyi ağırlıkları yükler.
model.load_state_dict(state); model.eval()                                     # Ağırlıkları uygular ve modeli değerlendirme moduna alır.

# ====== KISA YARDIMCILAR ======
def as_windows_flat(angles: np.ndarray) -> np.ndarray:
    """Açı dizisini (T,2) iken pencereleyip (N,W,2) ve düzleştirip (N,W*2) modele hazır hale getirir."""
    wins = windowize(angles, WINDOW_SIZE, WINDOW_STRIDE)                        # Açı serisini sabit uzunluklu pencerelere böler.
    flat = flatten_windows(wins)                                                # Pencereleri tek vektör haline getirir.
    return flat                                                                 # Eğitimdeki giriş formatı ile aynı şekli döndürür.

def recon_mse_per_window(x_flat: np.ndarray) -> np.ndarray:
    """Düzleştirilmiş pencereler için pencere başına ortalama karesel hata (MSE) hesaplar."""
    if x_flat.shape[0] == 0:                                                    # Pencere yoksa boş dizi döndürür.
        return np.empty((0,), dtype=np.float32)
    with torch.no_grad():                                                       # Gradyan hesabı kapatılarak inference yapılır.
        xt = torch.from_numpy(x_flat.astype(np.float32)).to(DEVICE)             # Girdiyi tensöre çevirip cihaza taşır.
        xr = model(xt)                                                          # Modelden yeniden üretim çıktısını alır.
        mse = ((xr - xt)**2).mean(dim=1).cpu().numpy()                          # Her pencerenin MSE değerini hesaplar.
    return mse                                                                  # Pencere başına MSE serisini döndürür.

# ====== 1) EĞİTİM MSE DAĞILIMI (KALİBRASYON) ======
def build_mse_scale():
    """Eğitim verilerinden reconstruction MSE dağılımını çıkarır ve p5–p95 aralığını kalibrasyon olarak belirler."""
    mses = []                                                                   # Tüm eğitim pencerelerinin MSE’lerini tutar.
    for vid in range(1, 61):                                                    # 1..60 arası tüm eğitim videolarını dolaşır.
        f = os.path.join(DIR_TR_ANG, f"angles_{vid}.npy")                       # Açı dosyasının yolunu oluşturur.
        if not os.path.exists(f):                                               # Dosya yoksa bu videoyu atlar.
            continue
        ang = np.load(f)                                                        # (T,2) açı serisini yükler.
        x = as_windows_flat(ang)                                                # (N,W*2) düzleştirilmiş pencereleri oluşturur.
        if x.shape[0] == 0:                                                     # Pencere oluşmadıysa geçer.
            continue
        mses.append(recon_mse_per_window(x))                                    # Bu videonun pencere MSE’lerini listeye ekler.
    if not mses:                                                                # Eğitimden hiç MSE çıkmadıysa hata verir.
        raise RuntimeError("Eğitim MSE dağılımı boş; açı dosyaları yok veya pencereler oluşmadı.")
    mses = np.concatenate(mses)                                                 # Tüm videolardan gelen MSE’leri tek dizide toplar.
    p5, p95 = np.percentile(mses, [5, 95])                                      # Gürbüz kalibrasyon için 5. ve 95. yüzdelikleri alır.
    stats = dict(mean=float(mses.mean()), std=float(mses.std()), p5=float(p5), p95=float(p95))  # Özet istatistikleri hazırlar.
    return stats                                                                # İstatistikleri döndürür.

scale = build_mse_scale()                                                       # Kalibrasyon istatistiklerini hesaplar.
print("Eğitim MSE istatistikleri → mean={mean:.6f}  std={std:.6f}  p5={p5:.6f}  p95={p95:.6f}".format(**scale))  # Bilgiyi yazdırır.

def mse_to_similarity(avg_mse: float, p5: float, p95: float) -> float:
    """Ortalama MSE’yi p5–p95 aralığına göre 0–100 arası benzerlik yüzdesine dönüştürür."""
    if p95 <= p5:                                                               # Aykırı bir durumda güvenli bir dönüşüm uygular.
        return float(max(0.0, min(100.0, 100.0 * (1.0 - avg_mse / (p5 + 1e-8)))))  # Emniyet amaçlı basit ölçekleme kullanır.
    t = (avg_mse - p5) / (p95 - p5)                                             # p5’te 0, p95’te 1 olacak şekilde normalize eder.
    sim = 100.0 * (1.0 - np.clip(t, 0.0, 1.0))                                  # Küçük MSE → yüksek benzerlik olacak şekilde tersler.
    return float(sim)                                                           # 0–100 arası benzerlik skorunu döndürür.

# ====== 2) TEST VİDEOSU İÇİN BENZERLİK % ======
def similarity_of_video(video_path: str, save_angles: bool=True) -> dict:
    """Videodan açı çıkarır, pencereler ve MSE ortalamasını p5–p95’e göre % benzerliğe map eder."""
    if not os.path.exists(video_path):                                          # Video dosyası yoksa hata döndürür.
        return {"path": video_path, "error": "Video bulunamadı."}
    ang = extract_angles(video_path)                                            # (T,2) normalize açı serisini çıkarır.
    if save_angles:                                                             # İstenirse açıyı test klasörüne kaydeder.
        fname = os.path.splitext(os.path.basename(video_path))[0]
        np.save(os.path.join(DIR_TE_ANG, f"angles_{fname}.npy"), ang)
    x = as_windows_flat(ang)                                                    # (N,W*2) düzleştirilmiş pencere vektörlerini üretir.
    if x.shape[0] == 0:                                                         # Pencere oluşmadıysa video çok kısa olabilir.
        return {"path": video_path, "error": "Pencere oluşmadı (video çok kısa olabilir)."}
    mse_win = recon_mse_per_window(x)                                           # Pencere başı MSE değerlerini hesaplar.
    avg_mse = float(mse_win.mean())                                             # Pencerelerin ortalama MSE’sini alır.
    sim_pct = mse_to_similarity(avg_mse, scale["p5"], scale["p95"])             # Ortalamayı 0–100 benzerliğe çevirir.
    return {"path": video_path, "avg_mse": avg_mse, "similarity_percent": sim_pct, "windows": int(x.shape[0])}  # Sonucu döndürür.

# ====== 3) 61..64 TEST DOSYALARINI HESAPLA ======
def eval_batch_test(ids=(61,62,63,64)):
    """Belirtilen test id’leri için yüzdesel benzerlikleri yazdırır."""
    print(f"\n{'Video':>7} | {'AvgMSE':>10} | {'Benzerlik%':>11} | {'Pencere':>8}")  # Tablo başlığını yazar.
    print("-"*45)                                                                    # Ayırıcı çizgi basar.
    for vid in ids:                                                                   # Her test id’si için işlemi tekrarlar.
        vpath = os.path.join(BASE, "test_video", f"{vid}.mp4")                        # Test videosunun yolunu oluşturur.
        res = similarity_of_video(vpath, save_angles=True)                            # Yüzde benzerliği hesaplar.
        if "error" in res:                                                            # Hata varsa bildirir.
            print(f"{vid:7d} | {res['error']}")
        else:                                                                          # Hata yoksa metrikleri basar.
            print(f"{vid:7d} | {res['avg_mse']:10.6f} | {res['similarity_percent']:11.2f} | {res['windows']:8d}")

# Çalıştır:
eval_batch_test()                                                                     # 61–64 test videolarını puanlar ve yazdırır.


Eğitim MSE istatistikleri → mean=0.000084  std=0.000114  p5=0.000005  p95=0.000352

  Video |     AvgMSE |  Benzerlik% |  Pencere
---------------------------------------------
     61 |   0.000081 |       78.02 |       36
     62 |   0.000043 |       88.95 |       34
     63 |   0.164606 |        0.00 |       33
     64 |   0.000330 |        6.32 |       30
